In [1]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

In [2]:
import { RecursiveSet } from './Recursive-Set';
type Variable = string;
type Literal = Variable | ['¬', Variable];
type Clause = RecursiveSet<Literal>;

# Refutational Completeness of the Cut Rule

This notebook implements a number of procedures that are needed in our proof of the <em style="color:blue">refutational completeness</em> of the cut rule.

The function $\texttt{complement}(l)$ computes the *complement* of a literal $l$.
As we are only dealing with formulas in CNF that are written in set notation and these formulas do not contain $\top$ or $\bot$, a literal is either 
a propositional variable or the negation of a propositional variable.
Therefore, if $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [3]:
function complement(l: Literal): Literal {
    if (Array.isArray(l)) {
        return l[1];
    } else {
        return ['¬', l];
    }
}

In [4]:
complement('p');

[ '¬', 'p' ]


In [5]:
complement(['¬', 'p']);

p


The function $\texttt{extractVariable}(l)$ extracts the propositional variable from the literal $l$.
If $p$ is a propositional variable, we have the following: 

1. $\texttt{extractVariable}(p) = p$,
2. $\texttt{extractVariable}(\neg p) = p$.

In [6]:
function extractVariable(l: Literal): Variable {
    if (Array.isArray(l)) {
        return l[1];
    } else {
        return l;
    }
}

In [7]:
extractVariable('p');

p


In [8]:
extractVariable(['¬', 'p']);

p


The function $\texttt{collectsVariables}(M)$ takes a set of clauses $M$ as its input and computes the set of all propositional variables occurring in $M$.  The clauses in $M$ are represented as sets of literals.

In [9]:
function collectVariables(M: RecursiveSet<Clause>): RecursiveSet<Variable> {
    const vars = new RecursiveSet<Variable>();
    for (const clause of M) {
        for (const l of clause) {
            vars.add(extractVariable(l));
        }
    }
    return vars;
}

In [10]:
const C1: Clause = new RecursiveSet<Literal>('p', 'q', 'r');
const C2: Clause = new RecursiveSet<Literal>(['¬', 'p'], ['¬', 'q'], ['¬', 's']);
const M: RecursiveSet<Clause> = new RecursiveSet<Clause>(C1, C2);
console.log(collectVariables(M).toString());

{p, q, r, s}


Given two clauses $C_1$ and $C_2$ that are represented as sets of literals, the function `cutRule`$(C_1, C_2)$ computes the set of all clauses that can be derived from $C_1$ and $C_2$ using the *cut rule*.  In set notation, the cut rule is the following rule of inference:
$$
   \frac{\displaystyle \;C_1\cup \{l\} \quad C_2 \cup \bigl\{\overline{\,l\,}\bigr\}}{\displaystyle C_1 \cup C_2}
$$

In [11]:
function cutRule(C1: Clause, C2: Clause): RecursiveSet<Clause> {
    const result = new RecursiveSet<Clause>();
    for (const l of C1) {
        const comp = complement(l);
        if (C2.has(comp)) {
            const newC1 = C1.clone();
            newC1.remove(l);
            const newC2 = C2.clone();
            newC2.remove(comp);
            result.add(newC1.union(newC2));
        }
    }
    return result;
}

In [12]:
const C1: Clause = new RecursiveSet<Literal>(['¬', 'q'], ['¬', 'r']);
const C2: Clause = new RecursiveSet<Literal>('q', 'r');
cutRule(C1, C2);

{{q, ['¬', 'q']}, {r, ['¬', 'r']}}


In the expression `saturate(Clauses)` below, `Clauses` is a set of *clauses*, where each clause is a set of *literals*.  The call `saturate(Clauses)` computes the set of all clauses that can be derived from clauses in the set `Clauses` using the *cut rule*.  The function keeps applying the cut rule until either no new clauses can be derived, or the empty clause $\{\}$ is derived.  The resulting set of clauses is *saturated* in the following sense:  If $C_1$ and $C_2$ are clauses from the set `Clauses` and the clause $D$ can be derived from $C_1$ and $C_2$ via the cut rule, then $D \in \texttt{Clauses}$ or $\{\} \in \texttt{Clauses}$.

In [13]:
function saturate(Clauses: RecursiveSet<Clause>): RecursiveSet<Clause> {
    const currentClauses = Clauses.clone();
    while (true) {
        const sizeBefore = currentClauses.size;
        const newClauses = new RecursiveSet<Clause>();
        for (const C1 of currentClauses) {
            for (const C2 of currentClauses) {
                const derived = cutRule(C1, C2);
                for (const D of derived) {
                    if (D.isEmpty()) {
                        const conflict = new RecursiveSet<Clause>();
                        conflict.add(new RecursiveSet<Literal>());
                        return conflict;
                    }
                    newClauses.add(D);
                }
            }
        }
        for (const newC of newClauses) currentClauses.add(newC);
        if (currentClauses.size === sizeBefore) return currentClauses;
    }
}

In [14]:
const C1: Clause = new RecursiveSet<Literal>('p', 'q');
const C2: Clause = new RecursiveSet<Literal>(['¬', 'p']);
const C3: Clause = new RecursiveSet<Literal>(['¬', 'p'], ['¬', 'q']);
const Clauses: RecursiveSet<Clause> = new RecursiveSet<Clause>(C1, C2, C3);
console.log(saturate(Clauses));

{{p, q}, {q}, {['¬', 'p']}, {p, q}, {p, ['¬', 'p']}, {q, ['¬', 'q']}, {['¬', 'p'], ['¬', 'q']}, {['¬', 'p'], ['¬', 'q']}, {['¬', 'p']}}


The function $\texttt{findValuation}(\texttt{Clauses})$ takes a set of clauses as input.  The function tries to compute a variable interpretation that makes all of these clauses `True`.  If this attempt is successful, a set of literals is returned.  This set of literals does not contain  any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If $\texttt{Clauses}$ is unsatisfiable, `False` is returned.

In [15]:
function findValuation(Clauses: RecursiveSet<Clause>): false | RecursiveSet<Literal> {
  const Variables = collectVariables(Clauses);
  const saturatedClauses = saturate(Clauses);
  for (const c of saturatedClauses) {
    if (c.isEmpty()) return false;
  }
  const Literals = new RecursiveSet<Literal>();
  for (const p of Variables) {
    const compLiterals = new RecursiveSet<Literal>();
    for (const l of Literals) compLiterals.add(complement(l)); 
    let satisfiesCondition = false;
    const posP: Literal = p;
    for (const C of saturatedClauses) {
      if (C.has(posP)) {
        let restIsEvaluatedFalse = true;
        for (const l of C) {
          const isP = !Array.isArray(l) && l === p;
          if (!isP && !compLiterals.has(l)) {
            restIsEvaluatedFalse = false;
            break;
          }
        }
        if (restIsEvaluatedFalse) {
          satisfiesCondition = true;
          break;
        }
      }
    }
    if (satisfiesCondition) {
        Literals.add(posP);
    } else {
        Literals.add(['¬', p]);
    }
  }
  return Literals;
}

In [16]:
const C01: Clause = new RecursiveSet<Literal>('r', 'p', 's');
const C02: Clause = new RecursiveSet<Literal>('r', 's');
const C03: Clause = new RecursiveSet<Literal>('p', 'q', 's');
const C04: Clause = new RecursiveSet<Literal>(['¬', 'p'], ['¬', 'q']);
const C05: Clause = new RecursiveSet<Literal>(['¬', 'p'], 's', ['¬', 'r']);
const C06: Clause = new RecursiveSet<Literal>('p', ['¬', 'q'], 'r');
const C07: Clause = new RecursiveSet<Literal>(['¬', 'r'], ['¬', 's'], 'q');
const C08: Clause = new RecursiveSet<Literal>(['¬', 'p'], ['¬', 's']);
const C09: Clause = new RecursiveSet<Literal>('p', ['¬', 'r'], ['¬', 'q']);
const C10: Clause = new RecursiveSet<Literal>(['¬', 'p'], 'r', 'q', ['¬', 's']);
const C11: Clause = new RecursiveSet<Literal>('p', 'q', 'r', ['¬', 's']);

const Clauses1: RecursiveSet<Clause> = new RecursiveSet<Clause>(
  C01, C02, C03, C04, C05, C06, C07, C08, C09, C10
);
findValuation(Clauses1);

{s, ['¬', 'p'], ['¬', 'q'], ['¬', 'r']}


In [17]:
const Clauses2: RecursiveSet<Clause> = new RecursiveSet<Clause>(
  C01, C02, C03, C04, C05, C06, C07, C08, C09, C10, C11
);
findValuation(Clauses2);

false


In [18]:
saturate(Clauses2);

{∅}


UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

...

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type

UncaughtException: Error: Unexpected pending rebuildTimer
    at sys.setTimeout (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:111:19)
    at scheduleProgramUpdate (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122735:35)
    at onSourceFileChange (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122876:7)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:122868:56
    at cb (C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\dist\converter.js:184:13)
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\typescript.js:5798:9
    at C:\Users\t.neithoefer\AppData\Local\anaconda3\node_modules\tslab\node_modules\@tslab\typescript-for-tslab\lib\type